# Task 3: explicit KV-cached greedy decoding

Choose **Runtime > Change runtime type > GPU**, then run cells in order.
Upload `colab_kv_cache_bundle.zip`. The selected adapter must already exist
on Drive under `TrialGPT_Assignment2/qlora_v1/adapter` from fine-tuning.
This notebook does not train. Model weights may download if the runtime is new.

Two manual forward-pass decoders are compared against HF generation.
The 3 x 3 stress matrix uses input lengths 128/256/512 and output lengths
16/32/64, with three timed repetitions per path (54 trials).
Three complete validation prompts separately check EOS stopping.
Stress prompts are artificial fixed-token workloads, not accuracy examples.
No held-out test records are included. A token mismatch remains a failure.


## 1. Check GPU and upload the code bundle


In [ ]:
import hashlib, io, json, zipfile
from pathlib import Path
import torch
from google.colab import files

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before continuing.")
print("GPU:", torch.cuda.get_device_name(0))
print("Upload colab_kv_cache_bundle.zip")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one code bundle.")
PROJECT = Path("/content/trialgpt_kv_cache")
expected = set(('decoding.py', 'benchmark_kv_cache.py', 'kv_benchmark_report.py', 'test_decoding.py', 'test_kv_benchmark.py', 'KV_CACHE.md', 'baseline.py', 'prepare_model_inputs.py', 'eda.py', 'requirements-colab-finetuning.txt', 'requirements-colab-baseline.txt', 'requirements-model-inputs.txt', 'requirements-data.txt', 'configs/model_input_config.json', 'configs/tokenizer_lock.json', 'configs/kv_cache_config.json', 'data/splits/split_manifest.json', 'data/model_ready/validation.jsonl'))
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    if len(archive.namelist()) != len(expected) + 1 or set(archive.namelist()) != expected | {"bundle_manifest.json"}:
        raise ValueError("Unexpected bundle contents.")
    manifest = json.loads(archive.read("bundle_manifest.json"))
    if set(manifest["file_sha256"]) != expected:
        raise ValueError("Unexpected bundle manifest.")
    for name in expected:
        if hashlib.sha256(archive.read(name)).hexdigest() != manifest["file_sha256"][name]:
            raise ValueError(f"Checksum mismatch: {name}")
    PROJECT.mkdir(parents=True, exist_ok=True)
    archive.extractall(PROJECT)
print("Code bundle verified; held-out test content:", manifest["contains_test_content"])


## 2. Install the pinned environment and check the decoder

Setup removes the unused optional torchao backend, as in the working
fine-tuning notebook. Tests run real miniature Qwen3 forwards on CPU.
Their timings do not measure the full selected model.


In [ ]:
import subprocess, sys

def run_checked(arguments):
    with subprocess.Popen(arguments, cwd=PROJECT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
    if code:
        raise RuntimeError(f"Command failed ({code}); full output is above: {arguments}")

run_checked([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])
run_checked([sys.executable, "-m", "pip", "install", "--quiet", "-r",
             str(PROJECT / "requirements-colab-finetuning.txt")])
run_checked([sys.executable, "-m", "unittest", "-v", "test_decoding", "test_kv_benchmark"])


## 3. Connect Drive and locate the selected adapter


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
TRAINING = Path("/content/drive/MyDrive/TrialGPT_Assignment2/qlora_v1")
RUN = Path("/content/drive/MyDrive/TrialGPT_Assignment2/kv_cache_v1")
for relative in ("training_summary.json", "training_manifest.json", "adapter/adapter_model.safetensors"):
    if not (TRAINING / relative).is_file():
        raise FileNotFoundError(f"Missing {TRAINING / relative}. Use the Drive folder from your completed fine-tuning run.")
run_checked([sys.executable, "-u", "benchmark_kv_cache.py",
             "--training-output", str(TRAINING), "--prepare-only"])


## 4. Run correctness checks and repeated GPU measurements

Let this finish. Model loading and correctness tests are outside timed trials.
Prefill and decode measurements synchronize CUDA. Both paths use the same
adapter, 4-bit weights, SDPA backend, tokens and greedy rule. Each case/path
is warmed; path order alternates across repetitions. Raw trials are saved on
Drive immediately. Rerunning with identical settings resumes missing trials.

If the cell reports a token mismatch, inspect `correctness.json` and
`trials.jsonl`. Continue to the download cell to share diagnostics.
Do not edit or discard mismatches to claim equivalence.


In [ ]:
run_checked([sys.executable, "-u", "benchmark_kv_cache.py",
             "--training-output", str(TRAINING), "--output", str(RUN)])
print((RUN / "summary.md").read_text())


## 5. Inspect and download the benchmark results


In [ ]:
from IPython.display import Image, display

if (RUN / "summary.md").exists():
    print((RUN / "summary.md").read_text())
if (RUN / "timings.png").exists():
    display(Image(filename=str(RUN / "timings.png")))
if not RUN.exists():
    raise FileNotFoundError("No benchmark output folder exists yet.")
download = Path("/content/kv_cache_results.zip")
with zipfile.ZipFile(download, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in RUN.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(RUN).as_posix())
files.download(str(download))
